<a href="https://colab.research.google.com/github/shahanasyedismail/CSA1731-AI/blob/main/feed_forward_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def sigmoid_deriv(a):
    return a * (1 - a)  # a = sigmoid(z)


class NeuralNetwork:
    def __init__(self, layer_sizes, lr=0.5, seed=42):
        rng = np.random.default_rng(seed)
        self.lr = lr
        self.W = [rng.standard_normal((layer_sizes[i], layer_sizes[i + 1])) * 0.5
                  for i in range(len(layer_sizes) - 1)]
        self.b = [np.zeros((1, layer_sizes[i + 1]))
                  for i in range(len(layer_sizes) - 1)]

    def forward(self, X):
        activations = [X]
        for W, b in zip(self.W, self.b):
            activations.append(sigmoid(activations[-1] @ W + b))
        return activations

    def backward(self, activations, y):
        m = y.shape[0]
        delta = (activations[-1] - y) * sigmoid_deriv(activations[-1])
        for i in reversed(range(len(self.W))):
            dW = activations[i].T @ delta / m
            db = delta.sum(axis=0, keepdims=True) / m
            if i > 0:
                delta = (delta @ self.W[i].T) * sigmoid_deriv(activations[i])
            self.W[i] -= self.lr * dW
            self.b[i] -= self.lr * db

    def train(self, X, y, epochs=10000, verbose=2000):
        for epoch in range(epochs + 1):
            acts = self.forward(X)
            self.backward(acts, y)
            if verbose and epoch % verbose == 0:
                loss = np.mean((acts[-1] - y) ** 2)
                print(f"Epoch {epoch:5d}  loss = {loss:.5f}")

    def predict(self, X):
        return self.forward(X)[-1]


if __name__ == "__main__":
    # XOR problem
    X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y = np.array([[0], [1], [1], [0]])

    nn = NeuralNetwork([2, 4, 1], lr=0.5)
    nn.train(X, y, epochs=10000)

    print("\nPredictions:")
    for xi, pi in zip(X, nn.predict(X)):
        print(f"  {xi} -> {pi[0]:.3f} -> {int(pi[0] > 0.5)}")

Epoch     0  loss = 0.25506
Epoch  2000  loss = 0.18477
Epoch  4000  loss = 0.02557
Epoch  6000  loss = 0.00602
Epoch  8000  loss = 0.00311
Epoch 10000  loss = 0.00205

Predictions:
  [0 0] -> 0.039 -> 0
  [0 1] -> 0.958 -> 1
  [1 0] -> 0.956 -> 1
  [1 1] -> 0.055 -> 0
